# Step 2 - Quantum Computing applicants

Input: `qc_applicants_<ENV>.csv` written at the end of Step 0 (one row per family-applicant).

In [2]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

ENV = "PROD"
DATA = Path("outputs/step2_input") / f"qc_applicants_{ENV}.csv"

df = pd.read_csv(DATA)
print(f"{len(df):,} rows | {df['docdb_family_id'].nunique():,} families | {df['psn_id'].nunique():,} applicants (psn_id)")
df.head()

20,895 rows | 12,596 families | 6,056 applicants (psn_id)


,docdb_family_id,psn_id,psn_name,han_name,psn_sector,applicant_ctry,n_applicants,family_year,offices,n_offices,granted_any,family_size,fwd_citations,first_office
0,65919538,21112,▲復▼旦大学ＦＵＮＤＡＮ ＵＮＩＶＥＲＳＩＴＹ,▲復▼旦大学Ｆｕｎｄａｎ Ｕｎｉｖｅｒｓｉｔｙ,UNIVERSITY,NaN,5,2018,CN EP JP US WO,5,True,5,8,CN
1,76545393,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,1,2018,US,1,False,1,6,US
2,71102213,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,5,2018,CN EP JP US WO,5,True,5,10,WO
3,72515606,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LABORATORY INC,COMPANY,CA,1,2018,US,1,False,1,5,US
4,66663731,41455,1QB INFORMATION TECHNOLOGIES,1QB INFORMATION TECHNOLOGIES INC.,COMPANY,NaN,4,2017,CA CN EP JP US WO,6,True,7,45,WO


## 1. Applicant type

PATSTAT `psn_sector` is used when it is known. Mixed sectors go to the public type: any `UNIVERSITY` -> university, any `GOV` -> gov / non-profit.

About half of the applicants are `UNKNOWN` or empty: they are mostly recent names that PATSTAT has not classified yet (Origin Quantum, Quantinuum, IQM, Pasqal ...). Their type is assigned from the name, with the first rule that matches:

1. a legal form (`LTD`, `INC`, `GMBH`, `有限公司` ...) -> **company** (university spin-offs are firms)
2. a university word (`UNIV`, `COLLEGE`, `ECOLE`, `大学` ...) -> **university**
3. a public research / government word (`INSTITUTE`, `ACADEMY`, `LABORATORY`, `CENTER`, `NATIONAL`, `MINISTRY` ...) -> **gov / non-profit**
4. a business word (`TECHNOLOGY`, `QUANTUM`, `SYSTEMS`, `&`, `科技` ...) -> **company**
5. `SURNAME, Name`, or 2-4 words made only of letters -> **individual**
6. a single word -> **company** (a brand: `PASQAL`, `THALES`)
7. anything else -> **uncertain**

A family can have applicants of different types (e.g. a company and a university). In the table of types it is counted **once in each type** it has, so that column sums to more than the number of families; the first printout counts every family once. At this stage families are counted in every applicant type they have, not fractionally: before name harmonisation one organisation can appear under several psn_id, so splitting a family among its applicants would give too much weight to organisations with many spellings. Fractional counting is applied after harmonisation.

In [3]:
LEGAL = ["INC", "LLC", "LLLP", "LL", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "COPMANY", "GMBH", "AG", "SE",
         "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "NA", "OY", "AB", "AS", "ASA", "PTY", "PLC", "KK", "LLP", "LP",
         "PTE", "KG", "OOO", "SP Z OO", "SPOLKA", "JOINT.STOCK", "ANONIM SIRKETI", "OBSHCHESTVO"]
LEGAL_CJK = ["有限公司", "株式会社", "股份", "公司", "会社", "주식회사", "インコーポレイテッド", "アクチボラグ"]
UNIV = ["UNIV", "COLLEGE", "POLYTECHN", "POLITECNIC", "ECOLE", "HOCHSCHULE", "SCHOOL", "INSTITUTE OF TECHNOLOGY", "GIJUKU",
        "TRUSTEES", "EDUCATIONAL FUND", "大学", "大學", "学院", "カレッジ", "義塾", r"\bETH\b", r"\bEPFL\b", r"\bUESTC\b"]
GOV = ["INSTIT", "INSTYTUT", "INSTUTUC", "ИНСТИТУТ", "ACADEM", "COMMISSARIAT", "CNRS", "LABORATORY", "CENTER", "CENTRE",
       "CENTRUM", "ZENTRUM", "FOUNDATION", "FONDAZIONE", "FUNDACIO", "STIFTUNG", "FRAUNHOFER", "FROUNHAV", "PLANCK",
       "GOVERNMENT", "MINISTRY", "MINISTERIUM", "BUNDES", "FEDERATSIYA", "AGENTSTVO", "UCHREZHDENIE", "SECRETARY", "AGENCY",
       "COUNCIL", "CONSEJO", "CONSIGLIO", r"\bNATIONAL\b", "IDRYMA", "ASSOCIATION", "HOSPITAL", "ARMY", "AIR FORCE", "NAVY",
       "RIKEN", r"\bTNO\b", "NASA", "研究", "中心", "委员会", "科学院", "センター"]
FIRM = ["TECHNOLOG", "QUANT", "COMPUT", "SYSTEM", "SOLUTION", r"\bLABS?\b", "SOFTWARE", "NETWORK", "ELECTRONIC", "SEMICONDUCT",
        "PHOTONIC", "DEVICES", "INDUSTR", "HOLDING", "GROUP", "CAPITAL", "VENTURES", "ENTERPRISE", "INTERNATIONAL", "SCIENCE",
        "DIGITAL", "DATA", "INTELLIGEN", "ENERGY", "PHARMA", "SECUR", "MACHINES", "INSTRUMENTS", "ENGINEERING", "INNOVATION",
        "BLOCKCHAIN", "BANK", "INSURANCE", "PORTFOLIO", r"\bAI\b", "&", r"\+", "科技", "技术", "量子"]

LEGAL_RE = re.compile(r"\b(" + "|".join(LEGAL) + r")\b|" + "|".join(LEGAL_CJK))
UNIV_RE, GOV_RE, FIRM_RE = (re.compile("|".join(words)) for words in (UNIV, GOV, FIRM))
WORD = r"[^\W\d_]+(?:['\-][^\W\d_]+)*"                              # letters only, any alphabet; O'NEIL, JEAN-PAUL
PERSON_COMMA = re.compile(rf"^{WORD}( {WORD}){{0,3}} ?, ?{WORD}")     # 'SURNAME, Name', 'VAN LAER, Raphael'
PERSON_WORDS = re.compile(rf"^{WORD}( {WORD}){{1,3}}$")               # 'LESCANNE Raphael', 'Taylor Lee Patti'


def normalise(name):
    "Upper case, no accents, full-width characters -> ASCII, no dots, no '?' (characters lost in PATSTAT)."
    s = unicodedata.normalize("NFKC", str(name))
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).upper()
    s = " ".join(s.replace("?", "").replace(".", "").split())
    return s.replace("UNIVERSITY CORPORATION", "UNIVERSITY")             # Japanese national universities


def type_from_name(name):
    n = normalise(name)
    if LEGAL_RE.search(n): return "company"
    if UNIV_RE.search(n):  return "university"
    if GOV_RE.search(n):   return "gov / non-profit"
    if FIRM_RE.search(n):  return "company"
    if PERSON_COMMA.match(n) or PERSON_WORDS.match(n) or "·" in n:   # '·' separates names in Chinese transliterations
        return "individual"
    if len(n.split()) == 1: return "company"
    return "uncertain"


def type_from_patstat(sector):
    s = str(sector).upper()
    if "UNIVERSITY" in s:             return "university"
    if "GOV" in s or "HOSPITAL" in s: return "gov / non-profit"
    if "COMPANY" in s:                return "company"
    if "INDIVIDUAL" in s:             return "individual"
    return None                                                        # UNKNOWN or empty


applicants = df.groupby("psn_id").agg(psn_name=("psn_name", "first"), psn_sector=("psn_sector", "first"),
                                      families=("docdb_family_id", "nunique"))
applicants["type"] = applicants["psn_sector"].map(type_from_patstat)
from_name = applicants["type"].isna()
applicants.loc[from_name, "type"] = applicants.loc[from_name, "psn_name"].map(type_from_name)
df["type"] = df["psn_id"].map(applicants["type"])

# ---- families: each family counted once ----
step0_ids = pd.read_csv(Path("outputs/step0") / f"qc_families_{ENV}.csv")["docdb_family_id"].unique()
types_of_family = df.groupby("docdb_family_id")["type"].agg(lambda s: sorted(set(s)))
n_types = types_of_family.str.len()
print(f"families in Step 0:                       {len(step0_ids):6,}")
print(f"  without any applicant in PATSTAT:       {len(step0_ids) - len(types_of_family):6,}   (dropped)")
print(f"  with at least one applicant:            {len(types_of_family):6,}")
for t in ["company", "university", "gov / non-profit", "individual", "uncertain"]:
    print(f"    only {t:<34}{(types_of_family.str.join('') == t).sum():6,}")
print(f"    2 or more types                       {(n_types > 1).sum():6,}")

# ---- applicants per type; a family is counted in every type it has ----
summary = pd.DataFrame({
    "applicants": applicants["type"].value_counts(),
    "of which from the name rule": applicants.loc[from_name, "type"].value_counts(),
    "families with at least one": df.groupby("type")["docdb_family_id"].nunique(),
}).fillna(0).astype(int).reindex(["company", "university", "gov / non-profit", "individual", "uncertain"])
display(summary)

uncertain = applicants[applicants["type"] == "uncertain"].sort_values("families", ascending=False)
print(f"{len(uncertain)} uncertain applicants:")
uncertain[["psn_name", "families"]]

families in Step 0:                       12,630
  without any applicant in PATSTAT:           34   (dropped)
  with at least one applicant:            12,596
    only company                            7,940
    only university                         2,337
    only gov / non-profit                     559
    only individual                           415
    only uncertain                              0
    2 or more types                        1,345


,applicants,of which from the name rule,families with at least one
type,,,
company,2598,1604,8977
university,855,243,2989
gov / non-profit,312,157,880
individual,2287,895,1181
uncertain,4,4,15


4 uncertain applicants:


,psn_name,families
psn_id,,
23866221,-NOT AVAILABLE-,12
6733642,DHITECH - DISTRETTO TECNOLOGICO H IGH TECH S.C...,1
7821231,EHNSEHRM (EHNSTITJU NAS'ONAL' DE LJA SANTE EH ...,1
187402713,PEARL COHEN ZEDEK LATZER BARATZ,1


## 2. Organisations only

Individuals are not harmonised and not counted: there is nothing to merge (different people with the same name would become fake giants such as `WANG`), and before the America Invents Act (Sept. 2012) US inventors were applicants by law, so keeping them would mix inventors with owners. The measures are therefore about **organisational ownership**: the 415 families with only individuals as applicants are lost (3.3%).

The three uncertain organisations are set by hand. `-NOT AVAILABLE-` is not an applicant: its 12 families all have other applicants, so only the row is dropped. The individuals found by the name rule with most families are printed as a check.

In [4]:
applicants.loc[[6733642, 187402713], "type"] = "company"   # DHITECH (consortium), PEARL COHEN (law firm)
applicants.loc[7821231, "type"] = "gov / non-profit"       # INSERM, transliterated
df["type"] = df["psn_id"].map(applicants["type"])

display(applicants[from_name & (applicants["type"] == "individual")].nlargest(10, "families")[["psn_name", "families"]])

org = df[(df["type"] != "individual") & (df["psn_name"] != "-NOT AVAILABLE-")].copy()
print(f"families:   {df['docdb_family_id'].nunique():,} -> {org['docdb_family_id'].nunique():,} with at least one organisation")
print(f"applicants: {df['psn_id'].nunique():,} -> {org['psn_id'].nunique():,} organisations (psn_id)")

,psn_name,families
psn_id,,
195262582,LESCANNE Rapha?l,7
192915572,Taylor Lee Patti,5
193942547,Elad Mentovich,5
194074734,Yonatan Piasetzky,5
194100481,Tali Septon,5
27646824,ROSE GEORDIE,4
186101699,"Dai, Haitao O.",4
189663297,Dr. Vishal Sharma,4
193314689,Eliahu Cohen,4


families:   12,596 -> 12,181 with at least one organisation
applicants: 6,056 -> 3,768 organisations (psn_id)


## 3. Name cleaning

Conservative cleaning, based on the dictionary method of the course (and Magerman et al. 2006): after cleaning, only names that are **identical** are merged, so nothing is merged by similarity. In order:

1. Unicode NFKC (full-width `ＮＴＴ` -> `NTT`), upper case, accents removed on Latin letters only (removing marks on Japanese kana would change the letter);
2. JPO markers `▲ ▼`, lost characters `?`, dots and apostrophes removed;
3. Japanese / Chinese names followed by their Latin version (`グーグル エルエルシーＧＯＯＧＬＥ ＬＬＣ`) -> the Latin part only;
4. `C/O ...` cut, `&` -> `AND`, any other symbol -> space (Chinese and Japanese letters are kept);
5. CJK legal forms removed at the start or end (`有限公司`, `株式会社` ...);
6. common words written in one way (`UNIVERSITY` -> `UNIV`, `TECHNOLOGIES` -> `TECH`, `LABORATORY` -> `LAB` ...), filler words removed (`OF`, `THE`, `AND` ...);
7. Latin legal forms removed at the start or end only (`INC`, `LLC`, `CO`, `GMBH`, `KABUSHIKI KAISHA` ...); if nothing is left the name is kept.

Families are counted once per clean name, also when two of its psn_id are on the same family.

In [5]:
FORMS = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "GMBH", "MBH",
         "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE",
         "KK", "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA"]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))     # longest first: 'CORPORATION' before 'CO'
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {"UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV", "UNIVERSIDAD": "UNIV",
         "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST", "INSTITUTO": "INST", "TECHNOLOGY": "TECH",
         "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH", "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB",
         "CENTRE": "CENTER", "SCIENCES": "SCIENCE", "大學": "大学"}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}


def clean(name):
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "\u0250")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)     # CJK name + Latin version -> Latin part
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"\bC/O\b.*$", "", s).replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    return re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s


org["clean"] = org["psn_name"].map(clean)
g = org.groupby("clean")
names = pd.DataFrame({"families": g["docdb_family_id"].nunique(), "psn_ids": g["psn_id"].nunique(),
                      "variants": g["psn_name"].agg(lambda s: " | ".join(s.unique()[:3]))}).sort_values("families", ascending=False)

share = names["families"].cumsum() / names["families"].sum()
print(f"{org['psn_id'].nunique():,} psn_id -> {len(names):,} clean names ({(names['psn_ids'] > 1).sum()} of them join 2+ psn_id)")
print("names with >= 5 / 10 / 20 families:", *[(names["families"] >= k).sum() for k in (5, 10, 20)])
print("share of applicant-family links in the top 10 / 50 / 100 / 200 names:", *[f"{share.iloc[n - 1]:.0%}" for n in (10, 50, 100, 200)])
pd.set_option("display.max_colwidth", 90)
display(names.head(30))
print("very short names (highest risk of a wrong merge):")
display(names[names.index.str.len() <= 3].head(10))


3,768 psn_id -> 2,651 clean names (470 of them join 2+ psn_id)
names with >= 5 / 10 / 20 families: 481 272 128
share of applicant-family links in the top 10 / 50 / 100 / 200 names: 22% 43% 54% 65%


,families,psn_ids,variants
clean,,,
IBM INTERNATIONAL BUSINESS MACHINES,772,1,IBM (INTERNATIONAL BUSINESS MACHINES CORPORATION)
ORIGIN QUANTUM COMPUTING TECH HEFEI,498,15,"Origin Quantum Computing Technology (Hefei) Co., Ltd | ORIGIN QUANTUM COMPUTING TECHNO..."
ORIGIN QUANTUM COMPUTING TECH,441,7,"ORIGIN QUANTUM COMPUTING TECHNOLOGY COMPANY | Origin Quantum Computing Technology Co.,..."
BEIJING BAIDU NETCOM SCIENCE TECH,332,3,BEIJING BAIDU NETCOM SCIENCE AND TECHNOLOGY COMPANY | BEIJING BAIDU NETCOM SCIENCE TEC...
GOOGLE,243,3,GOOGLE | グーグル エルエルシーＧＯＯＧＬＥ ＬＬＣ | GOOGLE LLC
MICROSOFT TECH LICENSING,241,5,"MICROSOFT TECHNOLOGY LICENSING | Microsoft Technology Licensing, LLC | Microsoft Techn..."
FUJITSU,227,1,FUJITSU
D WAVE SYSTEMS,213,6,D-WAVE SYSTEMS | D-WAVE SYSTEMS INC. | D-Wave Systems Inc.
国际商业机器,201,1,国际商业机器公司


very short names (highest risk of a wrong merge):


,families,psn_ids,variants
clean,,,
NEC,124,2,NEC CORPORATION | NEC Corporation
谷歌,86,1,谷歌有限责任公司
IBM,12,1,IBM CORPORATION
NTT,12,3,"NTT, Inc. | ＮＴＴ株式会社"
TDK,11,2,TDK CORPORATION | ＴＤＫ株式会社
英特尔,6,1,英特尔公司
康宁,5,1,康宁股份有限公司
GTN,4,1,GTN
TRW,4,1,TRW


No wrong merge among the very short names. Cleaning merges spellings of the **same name**; different names of the same organisation (translations, subsidiaries, renamings) are grouped in Section 4.

## 4. Manual grouping of the top applicants

The 128 clean names with at least 20 families (**anchors**, 58% of the applicant-family links) are grouped by hand. For each anchor the cell proposes candidates from all clean names: a **rare word** in common (in at most 10 names, so `QUANTUM`, `TECH`, `UNIV` do not count), the **same HAN name**, or a **similar spelling** (ratio >= 0.8).

Output: `candidates_top128.csv` (anchor, candidate, why). A candidate is in the same group (`same_group` = yes) if it is the same organisation or is majority-owned by it **at the filing date**: subsidiaries, IP holding companies and renamings yes; joint ventures, spin-offs and different universities of the same board no. Doubtful cases were checked with co-applicants and on the web.

In [6]:
from difflib import SequenceMatcher

anchors = names.index[names["families"] >= 20]                                    # sorted by families, largest first
words = pd.Series([w for n in names.index for w in set(n.split())]).value_counts()
rare = lambda n: {w for w in n.split() if words[w] <= 10 and len(w) >= 3 and not w.isdigit()}   # words found in few names only
han = org.groupby("clean")["han_name"].agg(set)

rows, seen = [], set()
for a in anchors:
    for c in names.index:
        why = [r for r, ok in [("rare word", rare(a) & rare(c)), ("same HAN", han[a] & han[c]),
                               ("similar", SequenceMatcher(None, a, c).ratio() >= 0.8)] if ok]
        if c == a or (why and (c in anchors or c not in seen)):      # a non-anchor name is proposed once, under the largest anchor
            rows.append([a, c, "ANCHOR" if c == a else ", ".join(why)])
            seen.add(c)

cand = pd.DataFrame(rows, columns=["anchor", "candidate", "why"])
cand["same_group"] = ""                                               # to fill: yes / no
cand["label"] = cand["anchor"].where(cand["why"] == "ANCHOR", "")     # group name, edit on ANCHOR rows only
cand["note"] = ""
OUT = Path("outputs/step2"); OUT.mkdir(parents=True, exist_ok=True)
cand.to_csv(OUT / "candidates_top128.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: Excel shows CJK correctly
print(f"{len(anchors)} anchors, {len(cand) - len(anchors)} candidates to check "
      f"({(cand.groupby('anchor').size() == 1).sum()} anchors have none)")
display(cand[cand["anchor"].isin(anchors[:2])])

128 anchors, 382 candidates to check (22 anchors have none)


,anchor,candidate,why,same_group,label,note
0,IBM INTERNATIONAL BUSINESS MACHINES,IBM INTERNATIONAL BUSINESS MACHINES,ANCHOR,,IBM INTERNATIONAL BUSINESS MACHINES,
1,IBM INTERNATIONAL BUSINESS MACHINES,IBM UNITED KINGDOM,rare word,,,
2,IBM INTERNATIONAL BUSINESS MACHINES,IBM DEUTSCHLAND,rare word,,,
3,IBM INTERNATIONAL BUSINESS MACHINES,IBM CHINA INVESTMENT,rare word,,,
4,IBM INTERNATIONAL BUSINESS MACHINES,QUANTUM MACHINES,rare word,,,
5,IBM INTERNATIONAL BUSINESS MACHINES,INTERNATIONAL BUSINESS MASCHINES,"same HAN, similar",,,
6,IBM INTERNATIONAL BUSINESS MACHINES,IBM,"rare word, same HAN",,,
7,IBM INTERNATIONAL BUSINESS MACHINES,IBM CHINA,rare word,,,
8,IBM INTERNATIONAL BUSINESS MACHINES,IBM ISRAEL SCIENCE TECH,rare word,,,
9,IBM INTERNATIONAL BUSINESS MACHINES,INTERNATIONAL BUSINESS MACHINES,"rare word, same HAN, similar",,,


The file was filled independently by three annotators: Claude (`valutazione_calude.csv`, which also holds the short group labels), Gemini (`valutazione_gemini.txt`) and ChatGPT (`valutazione_chat.csv`). Rows added by hand are not used. A pair listed under two anchors counts as one decision.

In [11]:
OUT = Path("outputs/step2")
FILES = {"claude": "valutazione_claude.csv", "gemini": "valutazione_gemini.txt", "chat": "valutazione_chat.csv"}   # one file per annotator


def read(f):
    d = pd.read_csv(OUT / f, dtype=str, sep=None, engine="python", encoding="utf-8-sig", comment="`").fillna("")
    d.columns = d.columns.str.strip()                                         # txt from a chatbot: separator guessed, ``` lines skipped
    d = d[~d["why"].isin(["ANCHOR", "added by hand"])]                       # rows added by hand: not compared
    return d.assign(pair=[" <> ".join(sorted(p)) for p in zip(d["anchor"], d["candidate"])])   # A>B and B>A = same decision

votes = pd.DataFrame({who: read(f)["same_group"].str.strip().str.lower().eq("yes").groupby(read(f)["pair"]).any()
                      for who, f in FILES.items()})
mine = pd.read_csv(OUT / FILES["claude"], dtype=str).fillna("")
labels = mine[mine["why"] == "ANCHOR"].set_index("anchor")["label"]           # short group names (Claude's file)
note = read(FILES["claude"]).groupby("pair")["note"].agg(lambda s: " ".join(x for x in s if x))

disagree = votes[votes.nunique(axis=1) > 1]
print(f"{len(votes)} pairs | all annotators agree on {1 - len(disagree) / len(votes):.1%} | {len(disagree)} to check by hand")
display(disagree.replace({True: "yes", False: "no"}).join(note.rename("Claude's note")))

352 pairs | all annotators agree on 96.6% | 12 to check by hand


,claude,gemini,chat,Claude's note
pair,,,,
ARIZONA BOARD REGENTS <> UNIV ARIZONA,no,yes,yes,Board of Regents on behalf of Arizona State Univ. (HAN): different university
ELEKTRON <> ELEQTRON,yes,yes,no,co-applicant on all its families with eleQtron: translation of the name
GUOKAI QIKE QUANTUM TECH BEIJING <> GUOKUNGKE QUANTUM TECH BEIJING,yes,no,yes,"garbled spelling of Guokai Qike, co-applicant with Qike Zhuhai"
HARBIN ENGINEERING UNIV <> HARBIN SCIENCE ENGINEERING UNIV,no,no,yes,
HARVARD COLLEGE LEADER LIKE <> HARVARD UNIV,yes,no,no,machine translation of 哈佛学院院长等 (President and Fellows of Harvard College)
HEWLETT PACKARD DEVELOPMENT <> HPE HEWLETT PACKARD ENTERPRISE,yes,yes,no,brand rule: HP and HPE share the brand after the Nov 2015 split -> same group (was: se...
JAPAN SCIENCE TECH AGENCY <> SCIENCE TECH AGENCY,no,yes,no,
NANJING REBORN QUANTUM TECH <> NANJING RUBAN QUANTUM TECH,yes,no,no,co-applicant with RUBAN QUANTUM TECH on all its families
PHOTONIC <> PHOTONIC TECH US,yes,no,yes,US subsidiary of Photonic Inc.


The pairs on which the annotators disagree are written to `disagreements.csv` and resolved by hand in `disagreements_final.csv` (column `final`).

In [12]:
# disagreements to resolve by hand: fill the 'final' column (yes / no)
disagree.replace({True: "yes", False: "no"}).join(note.rename("claude_note")).assign(final="").to_csv(OUT / "disagreements.csv", encoding="utf-8-sig")

Names still in Chinese, Japanese or Russian characters are listed in `translations.csv`. They were translated by hand in `translations_filled.csv`: `translation` (English name) and `group` (the existing Latin clean name of the same organisation, found also through co-applicants).

In [13]:
cjk = names[names.index.str.contains(r"[^\x00-\x7f]")]                 # clean names still in Chinese / Japanese / Korean characters
trans = cjk[["families", "variants"]].assign(translation="", group="", note="")   # translation: English name; group: existing label if any
trans.to_csv(OUT / "translations.csv", encoding="utf-8-sig")
print(f"{len(trans)} names to translate, {trans['families'].sum()} applicant-family links "
      f"({(trans['families'] >= 2).sum()} with 2+ families)")
display(trans.head(10))

113 names to translate, 544 applicant-family links (31 with 2+ families)


,families,variants,translation,group,note
clean,,,,,
国际商业机器,201,国际商业机器公司,,,
谷歌,86,谷歌有限责任公司,,,
微软技术许可,52,微软技术许可有限责任公司,,,
爱奥尼克,19,爱奥尼克公司,,,
耶鲁大学,11,耶鲁大学,,,
霍尼韦尔国际,10,霍尼韦尔国际公司,,,
特拉量子股份,8,特拉量子股份公司,,,
杜克大学,6,杜克大学,,,
英特尔,6,英特尔公司,,,


Linked names are grouped (union-find, so A-B and B-C give one group). Group name: the short label of the anchor, otherwise the largest clean name, otherwise the translation. Output: `groups_summary.csv`.


In [14]:
# final yes = unanimous yes or 'final' = yes; translations are linked to their Latin name
yes = votes.all(axis=1) | pd.read_csv(OUT / "disagreements_final.csv").set_index("pair")["final"].eq("yes").reindex(votes.index, fill_value=False)
tr = pd.read_csv(OUT / "translations_filled.csv", dtype=str).fillna("")
edges = [p.split(" <> ") for p in yes[yes].index] + [[a, b] for a, b in zip(tr["clean"], tr["group"]) if b]

parent = {n: n for n in names.index}                                          # union-find: linked names end in one group
def find(n):
    while parent[n] != n: n = parent[n]
    return n
for a, b in edges: parent[find(a)] = find(b)

lab = names[["families"]].assign(root=[find(n) for n in names.index], label=labels.reindex(names.index),
                                 tr=tr.set_index("clean")["translation"].reindex(names.index))
best = lab.rename_axis("clean").reset_index().sort_values("families", ascending=False).groupby("root").first()
best["name"] = best["label"].fillna(best["clean"].where(~best["clean"].str.contains(r"[^\x00-\x7f]"), best["tr"]))   # short label > largest name > translation
org["group"] = org["clean"].map(lab["root"]).map(best["name"])

g = org.groupby("group")
summary = pd.DataFrame({"families": g["docdb_family_id"].nunique(), "clean names": g["clean"].nunique(), "psn_id": g["psn_id"].nunique(),
                        "largest single name": org.groupby(["group", "clean"])["docdb_family_id"].nunique().groupby("group").max()})
summary = summary.sort_values("families", ascending=False)
summary.to_csv(OUT / "groups_summary.csv", encoding="utf-8-sig")
print(f"{org['psn_id'].nunique():,} psn_id -> {org['clean'].nunique():,} clean names -> {len(summary):,} groups "
      f"({(summary['clean names'] > 1).sum()} groups join 2+ names)")
display(summary.head(40))

3,768 psn_id -> 2,651 clean names -> 2,363 groups (147 groups join 2+ names)


,families,clean names,psn_id,largest single name
group,,,,
ORIGIN QUANTUM,1010,14,36,498
IBM,772,18,21,772
BAIDU,354,3,5,332
MICROSOFT,254,5,9,241
GOOGLE (ALPHABET),243,3,5,243
FUJITSU,229,4,5,227
D-WAVE,222,6,15,213
IONQ,173,3,16,168
INTEL,166,2,2,166


## 5. Typo merge in the tail

The tail is not checked by hand, so only typos are merged: names with the **same first word**, the **same digits** and at most **one different letter every 12** (spaces ignored), e.g. `AT T MOBLITY II` -> `AT T MOBILITY II`. The name with more families is kept.

Output: `typo_merges.csv`, reviewed by hand (column `ok`): the two US-government pairs (Army / Navy / Air Force) were set to `no`. The next cell applies the rest.

In [15]:
from itertools import combinations

fam = org.groupby("group")["docdb_family_id"].nunique()
latin = fam[~fam.index.str.contains(r"[^\x00-\x7f]")]


def typo_gap(a, b):
    "Characters that differ between two names, spaces ignored: 'MOBLITY' vs 'MOBILITY' = 1."
    a, b = a.replace(" ", ""), b.replace(" ", "")
    return len(a) + len(b) - 2 * sum(m.size for m in SequenceMatcher(None, a, b).get_matching_blocks())


rows = []
for _, block in latin.groupby(latin.index.str.split().str[0]):              # only names with the same first word are compared
    for a, b in combinations(block.index, 2):
        gap = typo_gap(a, b)
        if gap <= max(1, min(len(a), len(b)) // 12) and re.sub(r"\D", "", a) == re.sub(r"\D", "", b):   # 1 typo every 12 letters, same digits
            keep, merge = (a, b) if fam[a] >= fam[b] else (b, a)
            rows.append([keep, merge, fam[keep], fam[merge], gap])

typo = pd.DataFrame(rows, columns=["keep", "merge", "families keep", "families merge", "letters differing"]).assign(ok="yes")
typo.to_csv(OUT / "typo_merges.csv", index=False, encoding="utf-8-sig")
print(f"{len(typo)} proposed merges, {typo['families merge'].sum()} families moved; check the 'ok' column in typo_merges.csv")
display(typo.head(5))

30 proposed merges, 50 families moved; check the 'ok' column in typo_merges.csv


,keep,merge,families keep,families merge,letters differing,ok
0,ASSOCIATION POUR LA RECHERCHE LE DEVELOPPEMENT METHODES PROCESSUS INDUSTRIELS,ASSOCIATION POUR LA RECHERCHE LE DEVELOPPEMENTDE METHODES PROCESSUS INDUSTRIELS,1,1,2,yes
1,AT T MOBILITY II,AT T MOBLITY II,10,1,1,yes
2,BG NEGEV TECH APPLICATIONS LTD AT BEN GURION N UNIV,BG NEGEV TECH APPLICATIONS LTD AT BEN GURION UNIV,1,1,1,yes
3,DEUTSCHES ZENTRUM FUER LUFT RAUMFAHRT,DEUTSCHES ZENTRUM FUR LUFT RAUMFAHRT E V,1,1,3,yes
4,ECOLE NATIONALE SUPERIEURE MINES PARIS,ECOLE NATIONAL SUPERIEURE MINES PARIS,3,1,1,yes


In [16]:
# apply the merges with ok = yes
typo = pd.read_csv(OUT / "typo_merges.csv")
to = dict(zip(typo.loc[typo["ok"] == "yes", "merge"], typo.loc[typo["ok"] == "yes", "keep"]))


def follow(n):
    for _ in range(10): n = to.get(n, n)                                       # chains: A -> B -> C
    return n


org["group_3c"] = org["group"].map(follow)
print(f"{org['group'].nunique():,} groups -> {org['group_3c'].nunique():,} after typo merge")

2,363 groups -> 2,335 after typo merge


## 6. Acquisitions

Owner at filing: families filed from the **first full year after closing** go to the buyer, older ones stay with the target. Red Hat -> IBM (closed July 2019) and Cambridge Quantum -> Quantinuum (Nov 2021). Toshiba Memory needs no change (all its families are from before its sale in 2018).

In [17]:
# acquisitions: families filed from the first full year after closing go to the buyer (owner at filing)
ACQ = [("RED HAT", "IBM", 2020),                          # IBM closed the deal in July 2019
       ("CAMBRIDGE QUANTUM COMPUTING", "QUANTINUUM", 2022)]  # merged into Quantinuum in Nov 2021
org["group_final"] = org["group_3c"]
for target, buyer, year in ACQ:
    moved = (org["group_final"] == target) & (org["family_year"] >= year)
    org.loc[moved, "group_final"] = buyer
    print(f"{target} -> {buyer} from {year}: {org.loc[moved, 'docdb_family_id'].nunique()} families")

RED HAT -> IBM from 2020: 111 families
CAMBRIDGE QUANTUM COMPUTING -> QUANTINUUM from 2022: 8 families


## 7. Effect of each step

Cleaning reduces the number of actors but hardly moves the top; manual grouping moves the top (the measures of concentration depend on it); the typo merge only acts on the tail. *Families of all actors* is larger than the number of families because a family with two actors counts twice.

In [18]:
# actors and families at each step (a family is counted once per actor)
levels = {"raw psn_id": "psn_id", "after cleaning": "clean", "after grouping": "group", "after typo merge": "group_3c",
          "after acquisitions": "group_final"}
steps = pd.DataFrame({lvl: {"actors": org[c].nunique(),
                            "families of top 10": org.groupby(c)["docdb_family_id"].nunique().nlargest(10).sum(),
                            "families of all actors": org.groupby(c)["docdb_family_id"].nunique().sum()}
                      for lvl, c in levels.items()}).T
display(steps.join(steps.diff().add_prefix("change in ")).fillna(0).astype(int))

,actors,families of top 10,families of all actors,change in actors,change in families of top 10,change in families of all actors
raw psn_id,3768,3258,17751,0,0,0
after cleaning,2651,3341,15464,-1117,83,-2287
after grouping,2363,3577,13953,-288,236,-1511
after typo merge,2335,3577,13929,-28,0,-24
after acquisitions,2335,3688,13923,0,111,-6


## 8. Fractional counting

Each family counts 1 and is split equally among its **distinct** actors (organisations only: a family of a company and an inventor gives 1 to the company). The sum gives back the 12,181 families. `fa` (one row per family-actor, with weight, type and year) is the input of the analyses; `actors_fractional.csv` has full and fractional families per actor.

In [19]:
# fractional counting: each family = 1, split equally among its distinct actors (organisations only)
fa = org.drop_duplicates(["docdb_family_id", "group_final"])[["docdb_family_id", "group_final", "type", "family_year"]].copy()
fa["weight"] = 1 / fa.groupby("docdb_family_id")["group_final"].transform("size")

actors = fa.groupby("group_final").agg(families_full=("docdb_family_id", "size"), families_frac=("weight", "sum"))
actors = actors.sort_values("families_frac", ascending=False)
actors.to_csv(OUT / "actors_fractional.csv", encoding="utf-8-sig")

print(f"families: {fa['docdb_family_id'].nunique():,} | sum of fractional counts: {fa['weight'].sum():,.0f} | actors: {len(actors):,}")
display(actors.head(15).round(1))

families: 12,181 | sum of fractional counts: 12,181 | actors: 2,335


,families_full,families_frac
group_final,,
ORIGIN QUANTUM,1010,1008.0
IBM,883,872.0
BAIDU,354,354.0
MICROSOFT,254,252.5
GOOGLE (ALPHABET),243,243.0
D-WAVE,222,216.0
FUJITSU,229,213.8
INTEL,166,163.8
RUBAN QUANTUM,153,153.0


## Files (`outputs/step2/`)

| file | content |
|---|---|
| `candidates_top128.csv` | candidates for the 128 anchors (written) |
| `valutazione_claude.csv`, `valutazione_gemini.txt`, `valutazione_chat.csv` | the three filled versions (read) |
| `disagreements.csv` / `disagreements_final.csv` | disagreements (written) / resolved by hand (read) |
| `translations.csv` / `translations_filled.csv` | CJK names (written) / translated by hand (read) |
| `groups_summary.csv` | groups after manual grouping (written) |
| `typo_merges.csv` | proposed typo merges (written, reviewed by hand, read) |
| `actors_fractional.csv` | full and fractional families per actor (written) |

In [20]:
org[["psn_id", "psn_name", "clean", "group_final"]].to_csv("org_groups.csv", index=False, encoding="utf-8-sig")